# basic ML pipeline for weather forecasting
## notes
### Wind
speed (m/s)
deg -> direction, where 0\deg is due north
gust -> max gust

In [1]:
import requests
import os
import polars as pl
# import json
# from IPython.display import Image, display
# from io import BytesIO
from dotenv import load_dotenv

load_dotenv("../.env.local")

OWM_API_KEY = os.environ.get("OPENWEATHERMAP_API")

In [ ]:
def free_5d_forecast_from_zip(city):
    weather_url = f"https://api.openweathermap.org/data/2.5/forecast?q={city}&units=metric&APPID={OWM_API_KEY}"
    result = requests.get(weather_url)
    if result.status_code != 200:
        print(f'Access for {city} failed with {result.status_code}')
        print(result.text)
    return result.json()

    
def OWM_coord_forecast(coords: dict, units: str = "metric", exclude: list = []) -> dict:
    base_url = "https://api.openweathermap.org/data/3.0/onecall"
    required_parameters = f"appid={OWM_API_KEY}&lat={coords['lat']}&lon={coords['lon']}"

    exclude_result = "" if len(exclude) == 0 else f"?exclude={",".join(exclude)}"
    units_result = f"?units={units}"

    built_url = f"{base_url}?{required_parameters}" + units_result + exclude_result
    print(f"Calling: {built_url}")

    # res = requests.get(built_url)
    return built_url  # res.json()


def get_weather_icon(icon_id, scale: int = 1):
    # img = Image(icon.content)
    # display(img)
    url = f"https://openweathermap.org/img/wn/{icon_id}@2x.png"
    icon = requests.get(url)
    return icon

In [28]:
def geocode_from_zip(zip):
    location = f"{zip},US"
    geocode_url = f"https://api.openweathermap.org/geo/1.0/zip?zip={location}&APPID={OWM_API_KEY}"
    result = requests.get(geocode_url)
    if result.status_code != 200:
        print(f'Access for {zip} failed with {result.status_code}')
        print(result.text)
    return result.json()


def forecast_5d_from_coord(lat, lon):
    weather_url = f"https://api.openweathermap.org/data/2.5/forecast?appid={OWM_API_KEY}&lat={lat}&lon={lon}&units=metric"
    result = requests.get(weather_url)
    if result.status_code != 200:
        print(f'Access for lat: {lat}, lon: {lon} failed with {result.status_code}')
        print(result.text)
    return result.json()


def reshape_city_forecast(forecast):
    fc_city = forecast["city"]
    fc_threehourly = forecast["list"]

    fc_df = pl.LazyFrame(fc_threehourly)

    fc_df_time = pl.LazyFrame(fc_threehourly).select(
        "dt_txt",
        pl.from_epoch(pl.col("dt"), time_unit="s").sort()
    )

    location_df = pl.LazyFrame(fc_city).select(pl.all(), pl.col("coord").struct.unnest()).drop("coord").select(
        pl.col("id").alias("city_id"),
        pl.col("name").alias("city_name"),
        "population",
        "lat",
        "lon"
    )

    return fc_df.select(
        pl.from_epoch(pl.col("dt"), time_unit="s").set_sorted(),
        pl.col("dt_txt"),
        pl.col("sys").struct.field("pod").name.replace("pod", "part_of_day").cast(pl.Categorical),
        pl.col("main").struct.unnest(),
        pl.col("wind").name.prefix_fields("wind_").struct.unnest(),
        pl.col("visibility").alias("vis_meters"),
        pl.col("clouds").struct.unnest().name.replace("all", "cloud_cover_pct"),
        pl.col("pop").alias("p_of_precipitation").cast(pl.Float64),
        pl.col("weather").list.explode().struct.field("description").alias("weather_desc").cast(pl.Categorical)
    ).drop(["temp_kf", "pressure"]).join(fc_df_time.join(location_df, how="cross"), how="inner", on="dt")

In [30]:
list_of_cities = ["02472", "01907", "01945", "01940", "60606"]
# forecasts = free_5d_forecast_from_zip(list_of_cities[0])

coord_list = [{"lat": res["lat"], "lon": res["lon"]} for res in [geocode_from_zip(zip) for zip in list_of_cities]]

forecasts: pl.LazyFrame = [reshape_city_forecast(forecast_5d_from_coord(coord["lat"], coord["lon"])) for coord in coord_list]

result_frame = pl.LazyFrame(schema=forecasts[0].collect_schema())

for fc in forecasts:
    result_frame = pl.concat([result_frame, fc])

weather_forecasts_df = result_frame.drop("dt_txt_right").collect()

dt,dt_txt,part_of_day,temp,feels_like,temp_min,temp_max,sea_level,grnd_level,humidity,wind_speed,wind_deg,wind_gust,vis_meters,cloud_cover_pct,p_of_precipitation,weather_desc,city_id,city_name,population,lat,lon
datetime[μs],str,cat,f64,f64,f64,f64,i64,i64,i64,f64,i64,f64,i64,i64,f64,cat,i64,str,i64,f64,f64
2026-01-19 00:00:00,"""2026-01-19 00:00:00""","""n""",0.73,-1.54,0.47,0.73,1012,1005,91,1.95,43,4.41,189,100,0.87,"""light snow""",4954611,"""Watertown""",31915,42.37,-71.1773
2026-01-19 03:00:00,"""2026-01-19 03:00:00""","""n""",0.45,-2.18,0.24,0.45,1010,1003,95,2.21,16,6.76,55,100,1.0,"""snow""",4954611,"""Watertown""",31915,42.37,-71.1773
2026-01-19 06:00:00,"""2026-01-19 06:00:00""","""n""",-0.18,-3.25,-0.18,-0.18,1007,1001,98,2.51,359,6.95,116,100,1.0,"""snow""",4954611,"""Watertown""",31915,42.37,-71.1773
2026-01-19 09:00:00,"""2026-01-19 09:00:00""","""n""",-0.73,-2.59,-0.73,-0.73,1008,1002,98,1.5,330,3.54,183,100,1.0,"""light snow""",4954611,"""Watertown""",31915,42.37,-71.1773
2026-01-19 12:00:00,"""2026-01-19 12:00:00""","""n""",-0.84,-0.84,-0.84,-0.84,1008,1003,98,0.6,260,1.18,134,100,1.0,"""light snow""",4954611,"""Watertown""",31915,42.37,-71.1773
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-01-23 09:00:00,"""2026-01-23 09:00:00""","""n""",-12.54,-19.54,-12.54,-12.54,1030,1006,67,5.57,301,10.01,10000,100,0.0,"""overcast clouds""",4887398,"""Chicago""",2695598,41.8868,-87.6386
2026-01-23 12:00:00,"""2026-01-23 12:00:00""","""n""",-15.42,-22.42,-15.42,-15.42,1033,1009,67,6.19,311,9.54,10000,100,0.0,"""overcast clouds""",4887398,"""Chicago""",2695598,41.8868,-87.6386
2026-01-23 15:00:00,"""2026-01-23 15:00:00""","""d""",-17.31,-24.31,-17.31,-17.31,1037,1012,66,5.96,320,7.82,10000,76,0.0,"""broken clouds""",4887398,"""Chicago""",2695598,41.8868,-87.6386
